In [2]:
import numpy as np
import pandas as pd


# 1) min-max normalize
def minmax_normalize(series):
    s = series.astype(float).copy()
    s_min = s.min()
    s_max = s.max()

    if pd.isna(s_min) or pd.isna(s_max):
        return pd.Series(np.zeros(len(s)), index=s.index)

    if s_max == s_min:
        return pd.Series(np.zeros(len(s)), index=s.index)

    return (s - s_min) / (s_max - s_min)


# 2) build item-level signals from synthetic history
def build_item_signal_table(
    synthetic_view_history,
    item_col="title",
    save_col="save",
    listen_ratio_col="listen_ratio"
):
    df = synthetic_view_history.copy()

    df["save_binary"] = (df[save_col] == "yes").astype(int)
    df[listen_ratio_col] = pd.to_numeric(df[listen_ratio_col], errors="coerce")

    item_stats = (
        df.groupby(item_col, as_index=False)
          .agg(
              plays=(item_col, "size"),
              avg_listen_ratio=(listen_ratio_col, "mean"),
              save_rate=("save_binary", "mean")
          )
    )

    return item_stats


# 3) compute exposure / quality / inclusion signals
def compute_exposure_fairness_signals(
    item_stats,
    item_col="title",
    alpha=0.7
):
    """
    alpha = weight for listen ratio
    (1 - alpha) = weight for save rate
    """
    df = item_stats.copy()

    # Exposure: normalized play count
    df["Exp_i"] = minmax_normalize(df["plays"])

    # Normalize engagement signals
    df["L_norm_i"] = minmax_normalize(df["avg_listen_ratio"])
    df["S_norm_i"] = minmax_normalize(df["save_rate"])

    # Simplified quality proxy
    df["Q_i"] = alpha * df["L_norm_i"] + (1 - alpha) * df["S_norm_i"]

    # Inclusion signal
    df["I_i"] = (1 - df["Exp_i"]) * df["Q_i"]

    return df[[item_col, "plays", "avg_listen_ratio", "save_rate",
               "Exp_i", "L_norm_i", "S_norm_i", "Q_i", "I_i"]]


# 4) rerank baseline recommendations
def rerank_with_exposure_fairness(
    baseline_recs,
    item_signal_table,
    user_col="user_id",
    item_col="title",
    score_col="baseline_score",
    lambda_=0.6,
    theta_r=0.3,
    top_k=10
):
    """
    baseline_recs expected columns:
    - user_id
    - title (or item key)
    - baseline_score
    """
    recs = baseline_recs.copy()
    signals = item_signal_table.copy()

    # merge item-level fairness signals onto baseline recs
    merged = recs.merge(signals, on=item_col, how="left")

    # if some items have no signal yet, fill with 0
    for col in ["Exp_i", "L_norm_i", "S_norm_i", "Q_i", "I_i"]:
        if col in merged.columns:
            merged[col] = merged[col].fillna(0)

    # relevance safeguard
    merged["eligible"] = merged[score_col] >= theta_r

    # final score
    merged["final_score"] = np.where(
        merged["eligible"],
        lambda_ * merged[score_col] + (1 - lambda_) * merged["I_i"],
        merged[score_col]
    )

    # sort within each user
    merged = merged.sort_values(
        by=[user_col, "final_score"],
        ascending=[True, False]
    )

    # assign reranked position
    merged["rerank_position"] = merged.groupby(user_col).cumcount() + 1

    if top_k is not None:
        merged = merged[merged["rerank_position"] <= top_k].copy()

    return merged

In [3]:
import pandas as pd

synthetic_view_history = pd.read_csv(
    "/Users/tricialee/Documents/MasterUU/Media/ASM/PreferenceSystem/INFOMPPM/synthetic_view_history_simple.csv"
)

item_stats = build_item_signal_table(
    synthetic_view_history,
    item_col="title",
    save_col="save",
    listen_ratio_col="listen_ratio"
)

item_signal_table = compute_exposure_fairness_signals(
    item_stats,
    item_col="title",
    alpha=0.7
)

item_signal_table.head()

,title,plays,avg_listen_ratio,save_rate,Exp_i,L_norm_i,S_norm_i,Q_i,I_i
0,#CancelKarenDunbar,61,0.347049,0.098361,0.009098,0.224356,0.098361,0.186557,0.184860
1,'Go Back To Where You Came From',46,0.298913,0.021739,0.006823,0.155590,0.021739,0.115435,0.114647
2,"'Til Kingdom Come: Trump, Faith and Money",908,0.352081,0.055066,0.137528,0.231545,0.055066,0.178601,0.154039
3,"1Xtra - 1XTRA Talks: Pain, Power and Progress",393,0.814122,0.251908,0.059439,0.891603,0.251908,0.699695,0.658106
4,1Xtra - 1Xtra's Hot 4 2022,176,0.347443,0.039773,0.026535,0.224919,0.039773,0.169375,0.164881
